# Your Robot's First Steps

Welcome to *Autonomous Systems and Mobile Robots*. In this notebook you will build a minimal but complete autonomous agent: a point-robot that **perceives** its surroundings, **reasons** about what to do, and **acts** on that decision — the Perceive-Reason-Act (PRA) loop that will guide the entire course.

The scenario is intentionally simple: a 2-D grid world, four compass directions, and basic obstacle avoidance. The same conceptual loop, however, governs the quadruped you saw in the live demo — and every autonomous robot you will work with in the coming weeks.

### What you will do

| Part | PRA Phase | Task |
|------|-----------|------|
| 0 | — | Verify your Python / Jupyter environment |
| 1 | — | Represent a robot's state as $(x, y, \theta)$ |
| 2 | **Act** | Move the robot through discrete actions |
| 3 | **Perceive** | Sense obstacles in a grid world |
| 4 | **Reason + Loop** | Decide and close the PRA loop |
| 5 | Bonus | Open-ended: make the robot smarter |

**Prerequisites** — You should have worked through the introductory notebooks in `python-intro/` (`python_basics.ipynb`, `numpy.ipynb`, `matplotlib.ipynb`). This exercise assumes familiarity with basic Python syntax, NumPy arrays, and Matplotlib plotting.

**Time estimate** — approximately 45 minutes.

---

## Part 0 — Environment Check

Run the cell below. If it executes without errors and you see a small plot, your environment is ready. No exercise here — just verify.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils import (
    DIRECTION_VECTORS,
    FORWARD,
    TURN_LEFT,
    TURN_RIGHT,
    setup_matplotlib,
    draw_robot,
    draw_world,
    draw_trajectory,
    is_blocked,
)

# Quick sanity check
x = np.linspace(0, 2 * np.pi, 100)
plt.figure(figsize=(6, 2))
plt.plot(x, np.sin(x), label="sin(x)")
plt.title("Environment Check")
plt.legend()
plt.tight_layout()
plt.show()

print(f"NumPy version: {np.__version__}")
print("Environment is ready.")

### Matplotlib defaults

The helper below (imported from `utils.py`) sets reasonable plotting defaults used throughout the notebook.

In [ ]:
setup_matplotlib()

---

## Part 1 — Robot State: Where Am I?

Every robot — whether it walks on four legs, rolls on wheels, or flies — needs an internal representation of *where it is*. The simplest useful representation in 2-D is the **state vector**

$$\mathbf{s} = \begin{pmatrix} x \\ y \\ \theta \end{pmatrix}$$

where $(x, y)$ is the position on the grid and $\theta$ is the heading (orientation). For our discrete grid world we restrict $\theta \in \{0°, 90°, 180°, 270°\}$, corresponding to the four compass directions:

| $\theta$ | Direction | Effect on position |
|----------|-----------|---------------------|
| 0° | East | $x$ increases |
| 90° | North | $y$ **decreases** (moves up on screen) |
| 180° | West | $x$ decreases |
| 270° | South | $y$ **increases** (moves down on screen) |

> **Coordinate note** — the grid uses *screen coordinates*: `x` is the column (left→right) and `y` is the row (top→bottom). Row 0 is at the **top**, so moving "North" (upward on screen) means subtracting 1 from `y`. This matches how NumPy stores 2-D arrays: `array[y, x]`.

In the weeks ahead you will extend this to continuous poses, add uncertainty, and eventually work in 3-D. For now, this discrete formulation keeps the focus on the PRA concept itself.

### Visualization helpers

The utility functions for rendering robots and grid worlds (`draw_robot`, `draw_world`, `draw_trajectory`, `is_blocked`) are imported from `utils.py`. You can open that file if you want to read how they work, but you do **not** need to modify them.

<div class='alert alert-warning'>
<b>Exercise 1 — Create a robot state</b>

Complete the function below. It should return a NumPy array `[x, y, theta]`.
</div>

In [ ]:
def create_robot(x: int, y: int, theta: int) -> np.ndarray:
    """
    Create a robot state vector.

    Parameters
    ----------
    x : int
        Horizontal grid position.
    y : int
        Vertical grid position.
    theta : int
        Heading in degrees (0, 90, 180, or 270).

    Returns
    -------
    state : np.ndarray, shape (3,)
        The state vector [x, y, theta].
    """
    
    # ---- TODO: return a NumPy array containing x, y, and theta ----
    state = ...  # <-- replace this
    # ---- END TODO -------------------------------------------------
    
    return state

In [ ]:
# Test your function
s = create_robot(1, 2, 90)

assert isinstance(s, np.ndarray), "Return type must be np.ndarray"
assert s.shape == (3,), f"Expected shape (3,), got {s.shape}"
assert np.array_equal(s, [1, 2, 90]), f"Expected [1, 2, 90], got {s}"

print("Passed.")

<div class='alert alert-warning'>
<b>Exercise 2 — Visualize multiple robots</b>

Complete the function so that it draws every robot state on a shared grid plot.  
Use `draw_robot(state, ax)` inside a loop.
</div>

In [ ]:
def draw_multiple_robots(states: list[np.ndarray], grid_size: int = 7) -> None:
    """
    Draw several robots on a blank grid.

    Parameters
    ----------
    states : list of np.ndarray
        Each element is a state vector [x, y, theta].
    grid_size : int
        Side length of the square grid.
    """
    fig, ax = plt.subplots(figsize=(6, 6))
    blank_world = np.zeros((grid_size, grid_size), dtype=int)
    draw_world(blank_world, ax)

    # ---- TODO: loop over *states* and call draw_robot(state, ax) ----

    # ---- END TODO ---------------------------------------------------

    ax.set_title("Multiple robots on the grid")
    plt.tight_layout()
    plt.show()

In [ ]:
# Create a handful of robots at different positions and headings
robots = [
    create_robot(1, 1, 0),
    create_robot(3, 5, 90),
    create_robot(5, 2, 180),
    create_robot(4, 4, 270),
]
draw_multiple_robots(robots)

---

## Part 2 — Act: Making the Robot Move

A robot that knows where it is but cannot move is not particularly useful. In this section we implement the **Act** component of the PRA loop — the part that translates a chosen action into a change of state.

We define three discrete actions:

| Constant | Value | Effect |
|----------|-------|--------|
| `FORWARD` | 0 | Move one cell in the current heading direction |
| `TURN_LEFT` | 1 | Rotate heading by +90° (counter-clockwise) |
| `TURN_RIGHT` | 2 | Rotate heading by −90° (clockwise) |

The state transition is:

$$\mathbf{s}_{t+1} = f(\mathbf{s}_t, a_t)$$

For `FORWARD`, the new position is computed from the heading-dependent direction vector:

$$x_{t+1} = x_t + \Delta x(\theta_t), \qquad y_{t+1} = y_t + \Delta y(\theta_t)$$

For the turns, only the heading changes:

$$\theta_{t+1} = (\theta_t \pm 90°) \mod 360°$$

<div class='alert alert-warning'>
<b>Exercise 3 — Implement the act function</b>

Fill in the missing lines so that `act` returns a **new** state after applying the given action.  
Use `DIRECTION_VECTORS[theta]` to obtain $(\Delta x, \Delta y)$ for `FORWARD`.  
Use modular arithmetic `% 360` to keep $\theta$ in $\{0, 90, 180, 270\}$.
</div>

In [ ]:
def act(state: np.ndarray, action: int) -> np.ndarray:
    """
    Apply an action to the current state and return the new state.

    Parameters
    ----------
    state : np.ndarray, shape (3,)
        Current state [x, y, theta].
    action : int
        One of FORWARD (0), TURN_LEFT (1), TURN_RIGHT (2).

    Returns
    -------
    new_state : np.ndarray, shape (3,)
        Updated state after the action.
    """
    x, y, theta = state

    if action == FORWARD:
        dx, dy = DIRECTION_VECTORS[theta % 360]
        # ---- TODO: compute new_x and new_y ----
        new_x = ...  # <-- replace
        new_y = ...  # <-- replace
        # ---- END TODO -------------------------
        return np.array([new_x, new_y, theta])

    elif action == TURN_LEFT:
        # ---- TODO: compute new_theta (turn left = +90°, mod 360) ----
        new_theta = ...  # <-- replace
        # ---- END TODO -----------------------------------------------
        return np.array([x, y, new_theta])

    elif action == TURN_RIGHT:
        # ---- TODO: compute new_theta (turn right = -90°, mod 360) ----
        new_theta = ...  # <-- replace
        # ---- END TODO ------------------------------------------------
        return np.array([x, y, new_theta])

    else:
        raise ValueError(f"Unknown action: {action}")

In [ ]:
# Tests
s0 = create_robot(2, 3, 0)  # facing East

s1 = act(s0, FORWARD)
assert np.array_equal(s1, [3, 3, 0]), f"FORWARD failed: {s1}"

s2 = act(s0, TURN_LEFT)
assert np.array_equal(s2, [2, 3, 90]), f"TURN_LEFT failed: {s2}"

s3 = act(create_robot(2, 3, 0), TURN_RIGHT)
assert np.array_equal(s3, [2, 3, 270]), f"TURN_RIGHT failed: {s3}"

# Wrap-around check
s4 = act(create_robot(0, 0, 0), TURN_RIGHT)  # 0 - 90 = 270
assert s4[2] == 270, f"Wrap-around failed: {s4[2]}"

print("All act() tests passed.")

Let us verify visually. The cell below sends the robot on a short walk and plots its trajectory.

In [ ]:
# Drive a short command sequence and plot the trajectory
commands = [FORWARD, FORWARD, TURN_RIGHT, FORWARD, FORWARD, TURN_RIGHT, FORWARD]

state = create_robot(1, 1, 0)
trajectory = [state.copy()]
for cmd in commands:
    state = act(state, cmd)
    trajectory.append(state.copy())

fig, ax = plt.subplots(figsize=(6, 6))
blank = np.zeros((7, 7), dtype=int)
draw_trajectory(trajectory, blank, ax)
ax.set_title("Trajectory from a command sequence")
plt.tight_layout()
plt.show()

---

## Part 3 — Perceive: Sensing the World

A real robot uses sensors — LiDAR, cameras, sonar — to detect obstacles. Our simplified robot can check three adjacent cells: the one **ahead**, the one to its **left**, and the one to its **right**.

We represent the world as a 2-D NumPy array where

- `0` = free cell
- `1` = wall / obstacle

and the robot reads

$$\text{sensors} = \{\text{front}: b_f,\; \text{left}: b_l,\; \text{right}: b_r\}$$

with $b \in \{\texttt{True}, \texttt{False}\}$ indicating whether the cell is **blocked**.

Cells outside the grid boundary are treated as blocked.

### The grid world

Below we define a small maze. Walls are shown in dark grey. The robot will start in the top-left area and should ideally reach the bottom-right.

In [ ]:
# fmt: off
WORLD = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 1, 0, 1, 1, 0, 1],
    [1, 0, 1, 0, 0, 0, 0, 1, 0, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 1, 1, 1, 0, 1, 0, 1, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
], dtype=int)
# fmt: on

# The array maps directly: cell (x, y) corresponds to WORLD[y, x].
# (0, 0) is the top-left corner; y increases downward.

START = (1, 1)   # top-left corridor
GOAL  = (8, 8)   # bottom-right opening

fig, ax = plt.subplots(figsize=(6, 6))
draw_world(WORLD, ax, start=START, goal=GOAL)
ax.set_title("Grid World")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Helper: checking a grid cell

The function `is_blocked(x, y, world)` (imported from `utils.py`) returns `True` if a cell is blocked (wall or out of bounds).

<div class='alert alert-warning'>
<b>Exercise 4 — Implement the perceive function</b>

The robot can sense three adjacent cells: front, left, and right relative to its heading.  
The front direction is already computed for you. Complete the left and right directions.

**Hint**: If the robot faces East ($\theta = 0°$), then left is North ($\theta + 90°$) and right is South ($\theta - 90°$). Use `DIRECTION_VECTORS` and modular arithmetic, the same way you did in `act()`.
</div>

In [ ]:
def perceive(state: np.ndarray, world: np.ndarray) -> dict[str, bool]:
    """
    Return sensor readings for the three cells adjacent to the robot.

    Parameters
    ----------
    state : np.ndarray, shape (3,)
        Robot state [x, y, theta].
    world : np.ndarray, shape (H, W)
        Grid world.

    Returns
    -------
    sensors : dict
        {'front': bool, 'left': bool, 'right': bool}
        True means the cell is blocked.
    """
    x, y, theta = int(state[0]), int(state[1]), int(state[2])

    # Front direction — already done for you as an example
    fdx, fdy = DIRECTION_VECTORS[theta % 360]
    front_blocked = is_blocked(x + fdx, y + fdy, world)

    # ---- TODO: compute left direction vector ----
    # Turning left means rotating +90° counter-clockwise.
    # Example: facing East (0°) → left is North (90°)
    left_theta = ...  # <-- (theta + 90) % 360
    ldx, ldy = DIRECTION_VECTORS[left_theta]
    left_blocked = is_blocked(x + ldx, y + ldy, world)
    # ---- END TODO --------------------------------

    # ---- TODO: compute right direction vector ----
    # Turning right means rotating -90° clockwise.
    # Example: facing East (0°) → right is South (270°)
    right_theta = ...  # <-- (theta - 90) % 360
    rdx, rdy = DIRECTION_VECTORS[right_theta]
    right_blocked = is_blocked(x + rdx, y + rdy, world)
    # ---- END TODO ---------------------------------

    return {"front": front_blocked, "left": left_blocked, "right": right_blocked}

In [ ]:
# Tests — robot at (1, 1) facing East in the maze
test_state = create_robot(1, 1, 0)
sensors = perceive(test_state, WORLD)
print(f"Robot at (1,1) facing East: {sensors}")

assert isinstance(sensors, dict), "perceive must return a dict"
assert set(sensors.keys()) == {"front", "left", "right"}, "Keys must be front, left, right"

# Robot at (1, 2) facing North
s2 = create_robot(1, 2, 90)
sensors2 = perceive(s2, WORLD)
print(f"Robot at (1,2) facing North: {sensors2}")

print("perceive() structure tests passed.")

---

## Part 4 — Reason and Close the Loop

The final piece: given what the robot **perceives**, it must **reason** about which action to take. For now we use a deliberately simple policy — deterministic obstacle avoidance:

1. If the front is clear $\rightarrow$ go forward.
2. If the front is blocked and the right is clear $\rightarrow$ turn right.
3. Otherwise $\rightarrow$ turn left.

This is a reactive strategy: the robot has no memory and no planning. It is the simplest possible "brain", yet sufficient to navigate corridors. Later in the course you will learn proper planning algorithms.

### The complete PRA loop

At every time step $t$:

$$\text{sensors}_t = \texttt{perceive}(\mathbf{s}_t, \text{world})$$
$$a_t = \texttt{reason}(\text{sensors}_t)$$
$$\mathbf{s}_{t+1} = \texttt{act}(\mathbf{s}_t, a_t)$$

This is the heartbeat of every autonomous system you will build in this course.

<div class='alert alert-warning'>
<b>Exercise 5 — Implement the reason function</b>

Given the sensor dictionary, return an action. Apply the simple priority:
1. Front clear $\rightarrow$ `FORWARD`
2. Front blocked, right clear $\rightarrow$ `TURN_RIGHT`
3. Otherwise $\rightarrow$ `TURN_LEFT`
</div>

In [ ]:
def reason(sensors: dict[str, bool]) -> int:
    """
    Decide on an action given sensor readings.

    Parameters
    ----------
    sensors : dict
        {'front': bool, 'left': bool, 'right': bool}
        True = blocked.

    Returns
    -------
    action : int
        One of FORWARD, TURN_LEFT, TURN_RIGHT.
    """
    action: int
    
    # ---- TODO: implement the decision rules described above ----
    #
    # if front is clear:
    #     ...
    # elif front is blocked and right is clear:
    #     ...
    # else:
    #     ...
    #
    # ---- END TODO ----------------------------------------------

    return action

In [ ]:
# Tests
assert reason({"front": False, "left": False, "right": False}) == FORWARD
assert reason({"front": True, "left": True, "right": False}) == TURN_RIGHT
assert reason({"front": True, "left": False, "right": True}) == TURN_LEFT
assert reason({"front": True, "left": True, "right": True}) == TURN_LEFT
print("All reason() tests passed.")

<div class='alert alert-warning'>
<b>Exercise 6 — Close the PRA loop</b>

Connect your three functions inside a loop. At each step:
1. Perceive
2. Reason
3. Act (only if the resulting cell is free — skip the move otherwise)

Collect the trajectory and return it.
</div>

In [ ]:
def run_pra_loop(start_state: np.ndarray, world: np.ndarray, goal: tuple[int, int], n_steps: int = 50) -> list[np.ndarray]:
    """
    Execute the Perceive-Reason-Act loop for n_steps.

    Parameters
    ----------
    start_state : np.ndarray, shape (3,)
        Initial robot state.
    world : np.ndarray, shape (H, W)
        Grid world.
    goal : tuple (x, y)
        Target cell. The loop stops early when the robot reaches this cell.
    n_steps : int
        Maximum number of PRA iterations.

    Returns
    -------
    trajectory : list of np.ndarray
        Sequence of states (length <= n_steps + 1, including the start).
    """
    state = start_state.copy()
    trajectory = [state.copy()]

    for _ in range(n_steps):
        # Stop if the robot has reached the goal
        if int(state[0]) == goal[0] and int(state[1]) == goal[1]:
            break

        # ---- TODO: fill in the three PRA calls ----
        sensors = ...        # 1. Perceive
        action  = ...        # 2. Reason
        new_state = ...      # 3. Act
        # ---- END TODO -----------------------------

        # Safety check: only move FORWARD if the new cell is free.
        # Turns (TURN_LEFT / TURN_RIGHT) only change the heading θ —
        # the robot stays in its current cell — so they can never cause
        # a collision and are always accepted.
        nx, ny = int(new_state[0]), int(new_state[1])
        if not is_blocked(nx, ny, world):
            state = new_state
        else:
            # Action was a turn (position unchanged) — always accept
            if action != FORWARD:
                state = new_state

        trajectory.append(state.copy())

    return trajectory

In [ ]:
# Run the PRA loop
initial_state = create_robot(START[0], START[1], 0)  # Start facing East
trajectory = run_pra_loop(initial_state, WORLD, goal=GOAL, n_steps=80)

print(f"Trajectory length: {len(trajectory)} states")
print(f"Start : ({trajectory[0][0]:.0f}, {trajectory[0][1]:.0f})")
print(f"Final : ({trajectory[-1][0]:.0f}, {trajectory[-1][1]:.0f})")

> **Sanity check** — a correct implementation should reach the goal within roughly **30–50 steps**. If the trajectory never reaches the goal or runs for the full `n_steps` without stopping, re-check your `act()` function and the safety condition inside `run_pra_loop()`.

In [ ]:
# Visualize the full trajectory
fig, ax = plt.subplots(figsize=(7, 7))
draw_trajectory(trajectory, WORLD, ax, start=START, goal=GOAL)
ax.set_title("PRA Loop — Autonomous Navigation")
plt.tight_layout()
plt.show()

### Recap

You have just built a complete — if minimal — autonomous system:

- **Perceive**: the robot reads three binary sensor values from the grid.
- **Reason**: a simple reactive policy selects an action.
- **Act**: the state is updated deterministically.

Every week of this course adds sophistication to one or more of these components:

---

## Bonus — Make the Robot Smarter (Open Exercise)

> **This exercise is optional.** It is here for students who finish early or want an extra challenge. There is no single correct answer — experiment freely.

The reactive policy above is quite naive — the robot may loop in circles or get stuck in dead ends. Can you do better?

Below is an empty `reason_v2` function. Implement a smarter strategy and compare its trajectory to the baseline. Some ideas to explore:

- **Right-hand rule**: always keep your right hand on the wall. This is a classic maze-solving technique — the robot should prefer turning right and only turn left when absolutely necessary.
- **Memory**: give the robot a `visited` set and penalise revisiting cells.
- **Goal-seeking**: bias the action towards reducing the Manhattan distance $|x - x_g| + |y - y_g|$ to the goal, breaking ties with wall-following.

There is no single correct answer. Experiment, visualize, compare.

In [ ]:
def reason_v2(sensors: dict[str, bool], state: np.ndarray | None = None, memory: set | None = None) -> int:
    """
    A smarter reasoning policy. Design is up to you.

    Parameters
    ----------
    sensors : dict
        {'front': bool, 'left': bool, 'right': bool}
    state : np.ndarray or None
        Current state (optional, for goal-seeking strategies).
    memory : set or None
        Previously visited cells (optional, for memory-based strategies).

    Returns
    -------
    action : int
    """
    action: int
    
    # Your implementation here
    
    return action